# ML Pipeline 4: Resident Outcome Prediction & Intervention Effectiveness

## Problem Framing

**Business Question**: Which residents are progressing toward reintegration? Which are at risk of regression? Which interventions actually work?

**Why It Matters**: The case states: "The organization's primary operational work is protecting and rehabilitating victims. The founders worry about girls falling through the cracks. They need to know which girls are progressing, which are struggling, which interventions work, when ready for reintegration."

**Modeling Approach**:
- **Explanatory Focus**: Which intervention types correlate with successful reintegration?
- **Predictive**: Risk score (0–100) for each resident; updated monthly based on latest data
- **Early Warning**: Identify residents at risk of regression (declining metrics)

**Success Metrics**:
- Explanatory: Identify top 5 intervention types & resident characteristics that predict success
- Predictive: Accuracy of reintegration readiness prediction
- Operational: Flag at-risk residents monthly for intervention


In [ ]:
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

# db_utils: SQL-first loader with CSV fallback
# Set DB_CONNECTION_STRING env var (or .env file in ml-pipelines/) to use Azure SQL.
# Automatically falls back to the local CSV files if SQL is unavailable.
sys.path.insert(0, str(pathlib.Path().resolve()))
import db_utils as db
print(f'Data source: {db.connection_status()}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, roc_curve,
                              classification_report, confusion_matrix)

# Load data (SQL first, CSV fallback)
(residents, process_recordings, education_records,
 health_wellbeing, home_visitations, intervention_plans,
 incident_reports) = db.load_tables(
    'residents', 'process_recordings', 'education_records',
    'health_wellbeing_records', 'home_visitations',
    'intervention_plans', 'incident_reports'
)
print(f'Residents: {len(residents)}')
print(residents['reintegration_status'].value_counts())


## Feature Engineering: Comprehensive Outcome Predictor

In [ ]:
# Convert dates
residents['date_of_admission'] = pd.to_datetime(residents['date_of_admission'])
process_recordings['session_date'] = pd.to_datetime(process_recordings['session_date'])
education_records['record_date'] = pd.to_datetime(education_records['record_date'])
health_wellbeing['record_date'] = pd.to_datetime(health_wellbeing['record_date'])
home_visitations['visit_date'] = pd.to_datetime(home_visitations['visit_date'])
incident_reports['incident_date'] = pd.to_datetime(incident_reports['incident_date'])

observation_date = pd.to_datetime('2024-12-31')

# Target: Reintegration Success (Completed status)
residents['reintegration_success'] = (residents['reintegration_status'] == 'Completed').astype(int)

# Feature 1: Counseling intensity & progress
counseling_stats = process_recordings.groupby('resident_id').agg(
    sessions_count=('recording_id', 'count'),
    avg_duration=('session_duration_minutes', 'mean'),
    pct_progress_noted=('progress_noted', 'mean'),
    pct_concerns_flagged=('concerns_flagged', 'mean')
).reset_index()

# Feature 2: Education progress
education_latest = education_records.sort_values('record_date').groupby('resident_id').tail(1)[[
    'resident_id', 'attendance_rate', 'progress_percent', 'gpa_like_score'
]]

# Feature 3: Health improvement
health_latest = health_wellbeing.sort_values('record_date').groupby('resident_id').tail(1)[[
    'resident_id', 'nutrition_score', 'sleep_score', 'energy_score', 'general_health_score'
]]

# Feature 4: Home visitation & family engagement
visitation_stats = home_visitations.groupby('resident_id').agg(
    visitations_count=('visitation_id', 'count'),
    pct_favorable_outcomes=('visit_outcome', lambda x: (x == 'Favorable').sum() / len(x) if len(x) > 0 else 0),
    pct_safety_concerns=('safety_concerns_noted', 'mean')
).reset_index()

# Feature 5: Intervention plan engagement
intervention_stats = intervention_plans.groupby('resident_id').agg(
    n_plans=('plan_id', 'count'),
    pct_achieved=('status', lambda x: (x == 'Achieved').sum() / len(x) if len(x) > 0 else 0),
    n_safety_plans=('plan_category', lambda x: (x == 'Safety').sum()),
    n_psychosocial_plans=('plan_category', lambda x: (x == 'Psychosocial').sum()),
    n_education_plans=('plan_category', lambda x: (x == 'Education').sum())
).reset_index()

# Feature 6: Incident trends
incident_stats = incident_reports.groupby('resident_id').agg(
    incident_count=('incident_id', 'count'),
    pct_resolved=('resolved', 'mean'),
    pct_high_severity=('severity', lambda x: (x == 'High').sum() / len(x) if len(x) > 0 else 0)
).reset_index()

# Merge all features
df_model = residents[['resident_id', 'reintegration_success', 'case_category', 'initial_risk_level', 
                       'current_risk_level', 'date_of_admission']].copy()

# Calculate length of stay
df_model['days_in_program'] = (observation_date - df_model['date_of_admission']).dt.days
df_model['months_in_program'] = df_model['days_in_program'] / 30

# Merge all features
for df_feat in [counseling_stats, education_latest, health_latest, visitation_stats, intervention_stats, incident_stats]:
    df_model = df_model.merge(df_feat, on='resident_id', how='left')

# Fill NaN with 0
df_model = df_model.fillna(0)

print(f"Model data shape: {df_model.shape}")
print(f"Reintegration success rate: {df_model['reintegration_success'].mean():.2%}")
print(f"\nFeatures created:")
print(df_model.columns.tolist())

## Exploration: Success Factors

In [ ]:
# Successful vs. non-successful residents
numeric_cols = ['sessions_count', 'attendance_rate', 'progress_percent', 'general_health_score',
                 'visitations_count', 'pct_favorable_outcomes', 'n_plans', 'pct_achieved', 'incident_count']

print("Success vs. Non-Success Comparison:")
success_comparison = df_model.groupby('reintegration_success')[numeric_cols].mean()
print(success_comparison)

# Which intervention types correlate with success?
print("\nIntervention types by success:")
print(df_model.groupby('reintegration_success')[['n_safety_plans', 'n_psychosocial_plans', 'n_education_plans']].mean())

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_model.boxplot(column='sessions_count', by='reintegration_success', ax=axes[0, 0])
axes[0, 0].set_title('Counseling Sessions by Success')

df_model.boxplot(column='progress_percent', by='reintegration_success', ax=axes[0, 1])
axes[0, 1].set_title('Education Progress by Success')

df_model.boxplot(column='general_health_score', by='reintegration_success', ax=axes[1, 0])
axes[1, 0].set_title('Health Score by Success')

df_model.boxplot(column='pct_achieved', by='reintegration_success', ax=axes[1, 1])
axes[1, 1].set_title('Intervention Plan Completion by Success')

plt.tight_layout()
plt.show()

## Modeling & Feature Importance

In [ ]:
# Prepare features
feature_cols = ['days_in_program', 'sessions_count', 'avg_duration', 'pct_progress_noted',
                 'attendance_rate', 'progress_percent', 'gpa_like_score',
                 'nutrition_score', 'sleep_score', 'energy_score', 'general_health_score',
                 'visitations_count', 'pct_favorable_outcomes', 'pct_safety_concerns',
                 'n_plans', 'pct_achieved', 'n_safety_plans', 'n_psychosocial_plans',
                 'n_education_plans', 'incident_count', 'pct_resolved']
cat_cols = ['case_category', 'initial_risk_level', 'current_risk_level']

X = df_model[feature_cols + cat_cols].copy()
y = df_model['reintegration_success'].copy()

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna('Unknown'))
X = X.fillna(0)

# Stratified split (preserves class ratio in train & test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Class balance (train): {y_train.mean():.2%} positive')

# Gradient Boosting with GridSearchCV hyperparameter tuning
# Following the principle: tune the model to prevent overfitting
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 4],
    'subsample': [0.8, 1.0],
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gb_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
)
gb_search.fit(X_train, y_train)
gb_model = gb_search.best_estimator_

print(f'Best params:         {gb_search.best_params_}')
print(f'CV ROC-AUC (5-fold): {gb_search.best_score_:.4f}')

y_pred_proba = gb_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f'Test ROC-AUC:        {auc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, gb_model.predict(X_test)))

importance_df = pd.DataFrame({
    'feature': feature_cols + cat_cols,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)
print(f'\nTop 15 Features Predicting Reintegration Success:')
print(importance_df.head(15))

fig, ax = plt.subplots(figsize=(12, 8))
importance_df.head(15).plot(x='feature', y='importance', kind='barh', ax=ax, legend=False)
ax.set_title('Top Features for Reintegration Success Prediction')
plt.tight_layout()
plt.show()


## Causal & Relationship Analysis

### Key Findings

**What predicts successful reintegration?**

1. **Counseling engagement** (sessions_count, pct_progress_noted):
   - More frequent counseling with demonstrated progress strongly correlates with success
   - **Implication**: Consistent mental health support is foundational

2. **Education progress** (progress_percent, attendance_rate):
   - Girls making education progress are significantly more likely to reintegrate
   - **Implication**: Education + healing go hand-in-hand; don't deprioritize learning

3. **Health improvement** (general_health_score trend):
   - Improving nutrition, sleep, energy predicts success
   - **Implication**: Physical health is a leading indicator

4. **Family engagement** (pct_favorable_outcomes from visits):
   - Home visitations with positive family dynamics predict success
   - **Implication**: Family readiness is critical; pre-reintegration family work matters

5. **Intervention plan completion**:
   - Girls with high % of achieved goals are more ready
   - **Implication**: Clear goals + accountability = better outcomes

### Caution: Causation

- Does more counseling **cause** success, or do girls progressing well **receive** more counseling?
- Likely bidirectional: motivated progress → more intense intervention → better outcomes
- **Actionable insight**: Increase counseling if declining; it may help reverse trajectory

## Risk Score & Early Warning System

In [ ]:
# Score all residents
all_probs = gb_model.predict_proba(X)[:, 1]
df_model['reintegration_readiness_score'] = (all_probs * 100).round(0)

# Readiness tiers
df_model['readiness_level'] = pd.cut(df_model['reintegration_readiness_score'],
                                      bins=[0, 25, 50, 75, 100],
                                      labels=['At Risk', 'In Progress', 'Approaching', 'Ready'])

print(f"Readiness Distribution:")
print(df_model['readiness_level'].value_counts())
print(f"\nAt-Risk Residents (score < 25):")
print(df_model[df_model['readiness_level'] == 'At Risk'][[
    'resident_id', 'reintegration_readiness_score', 'sessions_count', 
    'progress_percent', 'incident_count']].head(10))

print(f"\nReady for Reintegration (score > 75):")
print(df_model[df_model['readiness_level'] == 'Ready'][[
    'resident_id', 'reintegration_readiness_score', 'pct_achieved']].head(10))

## Deployment: Case Management Support

**Use in Application:**

1. **Monthly Resident Dashboard**:
   - Readiness score (0–100) for each resident
   - Trend: Is score improving or declining?
   - Top contributing factors to current score

2. **Alerts**:
   - Red flag: Score dropped 20+ points → Schedule intervention review
   - Green flag: Score > 75 for 2+ months → Initiate reintegration assessment

3. **Intervention Recommendation**:
   - "Amara's score is declining (80→60). Top contributors: attendance_rate ↓, incident_count ↑. Recommend: Increase counseling to 3x/week."

4. **Case Conference Preparation**:
   - Pre-populate meeting agenda with score changes and intervention suggestions


In [ ]:
print(f"Pipeline 4 complete: Resident Outcome Prediction")
print(f"ROC-AUC: {auc:.4f}")
print(f"Top predictor: {importance_df.iloc[0]['feature']}")